# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiknaxTheGreek/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

### Research question

Can observable search, traffic, engagement, freshness and performance signals identify meaningful content opportunities or risks and help determine **which pages should be reviewed first, why they deserve attention, and what type of action should be considered?**

### Decision supported

The project supports a practical allocation decision: when a content, SEO or marketing team cannot inspect every page, which pages should enter a limited human-review queue first?

The intended product is therefore **decision support**, not automated content intervention. A reviewer receives a ranked queue with model risk, reason codes and a suggested diagnostic starting point, then decides whether any intervention is appropriate.

### Unit of analysis

One modeling row represents one pseudonymized content page at a fixed decision point. Predictors use information available by **31 March 2026**; the future outcome is observed during **April 2026** for that same page.

### Why the problem needs a baseline

A more complex model is only useful if it improves on a transparent rule under an honest validation design. For this reason the project freezes classification, regression and ranking baselines before model comparison, and treats **Precision@50** as the primary project metric because the operational output is a limited review queue.

The paper does not assume that ML must win. Negative or mixed out-of-sample results are retained as findings.

In [1]:
from pathlib import Path
import json
import pandas as pd

def repo_path(relative_path):
    """Resolve a repository-relative path whether run from repo root or notebook folder."""
    relative_path = Path(relative_path)
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not resolve repository path: {relative_path}")

with open(repo_path("work/outputs/assignment8_action_playbook_metrics.json"), "r", encoding="utf-8") as fh:
    playbook_receipt = json.load(fh)

question_contract = pd.DataFrame([
    {
        "item": "decision",
        "value": playbook_receipt["intended_use_and_limits"]["supported_decision"],
    },
    {
        "item": "intended_user",
        "value": playbook_receipt["intended_use_and_limits"]["intended_user"],
    },
    {
        "item": "operating_mode",
        "value": playbook_receipt["intended_use_and_limits"]["operating_mode"],
    },
    {
        "item": "final_decision_maker",
        "value": playbook_receipt["human_review_policy"]["final_decision_maker"],
    },
])

assert playbook_receipt["guardrails"]["human_review_required_for_every_row"] is True
assert playbook_receipt["guardrails"]["automatic_edit_or_publish_allowed"] is False
assert playbook_receipt["human_review_policy"]["final_decision_maker"] == "human_reviewer"

display(question_contract)
print("Question/decision contract verified from Assignment 8 receipts.")

,item,value
0,decision,prioritize a top-50 human-review queue and cho...
1,intended_user,content / SEO analyst or editor
2,operating_mode,research POC / human decision-support only
3,final_decision_maker,human_reviewer


Question/decision contract verified from Assignment 8 receipts.


## 2. Data

### Release and source

The analysis uses the pseudonymized **FlyRank internship warehouse** release, build `flyrank_pseudonymized_warehouse_release_v20260703`, exported on 3 July 2026 from the central data warehouse. The release documentation records **78,835,655 rows** in `fact_content_daily_performance`, whose grain is one daily content-performance observation per client and content item.

That figure describes the **warehouse scale**. It is not the model-training row count.

### Tables used

The final modeling contract uses:

- `fact_content_daily_performance` for March predictor construction and April outcome observations;
- `dim_content` for page creation date and therefore `content_age_days`;
- `dim_clients` for context only.

The fixed `fact_content_query_90d` table is deliberately excluded because its trailing window can overlap the future outcome period.

### Time windows

The decision cutoff is **31 March 2026**.

- Feature window: **1–31 March 2026**
- Outcome window: **1–30 April 2026**
- Minimum observability: at least **20 usable GSC days** in both months for an evaluable page.

The outcome compares average impressions per usable day rather than raw monthly totals so that the 31-day March window and 30-day April window are comparable.

### Final proof-of-concept population

The locked POC contains **2,520 pages from 21 pseudonymized clients**. To stop large clients or high-exposure pages from dominating, the sample is deliberately balanced:

- 840 Low-exposure pages;
- 840 Medium-exposure pages;
- 840 High-exposure pages.

This balancing improves the proof-of-concept comparison but changes the natural population mix. Therefore prevalence, calibration and aggregate rates from the 2,520-page sample must **not** be presented as natural FlyRank-wide prevalence.

### Observability and exclusions

The later audit found 106,546 March feature-eligible pages across 37 clients, of which 95,633 pages had sufficient April observability. That is **89.76%** of March-eligible pages.

The active contract excludes identifiers as predictive features, GA4 features because coverage is too sparse, absolute exposure as a predictor because exposure was already used for sampling strata, query-90d fields because of future-window risk, and mutable snapshot fields whose March-end value cannot be guaranteed.

All public reporting remains aggregate and pseudonymized; no client names, domains, URLs, page titles, keywords or raw search queries are exposed.

In [2]:
with open(repo_path("work/outputs/assignment7_leakage_audit.json"), "r", encoding="utf-8") as fh:
    leakage_audit = json.load(fh)

with open(repo_path("work/outputs/baseline_split_manifest.json"), "r", encoding="utf-8") as fh:
    split_manifest = json.load(fh)

guide_text = repo_path("docs/ml-intern-dataset-and-lane-guide.md").read_text(encoding="utf-8")

release_checks = {
    "warehouse_build_id_present": "flyrank_pseudonymized_warehouse_release_v20260703" in guide_text,
    "daily_fact_row_count_present": "78,835,655" in guide_text,
    "daily_fact_end_date_present": "2026-06-30" in guide_text,
}

population = leakage_audit["population_selection"]
data_summary = pd.DataFrame([
    {"measure": "POC pages", "value": split_manifest["population_pages"]},
    {"measure": "POC clients", "value": split_manifest["population_clients"]},
    {"measure": "March feature-eligible pages", "value": population["march_feature_eligible_pages"]},
    {"measure": "April-observable pages", "value": population["april_observable_pages"]},
    {
        "measure": "Outcome-observable share (%)",
        "value": round(population["pct_march_pages_retained_for_observable_outcome"], 2),
    },
])

assert all(release_checks.values()), release_checks
assert split_manifest["population_pages"] == 2520
assert split_manifest["population_clients"] == 21
assert split_manifest["train_pages"] + split_manifest["test_pages"] == 2520
assert split_manifest["client_overlap"] == []

display(data_summary)
print("Release metadata checks:", release_checks)
print("Balanced POC and observability contract verified.")

,measure,value
0,POC pages,2520.00
1,POC clients,21.00
2,March feature-eligible pages,106546.00
3,April-observable pages,95633.00
4,Outcome-observable share (%),89.76


Release metadata checks: {'warehouse_build_id_present': True, 'daily_fact_row_count_present': True, 'daily_fact_end_date_present': True}
Balanced POC and observability contract verified.


## 3. Methodology

### Target and task structure

The project uses three supervised components that support the final review queue.

**Classification** asks whether the page declines in the future:

`future_decline = 1 if future_impression_change < 0 else 0`

**Regression** estimates the size of future movement:

`future_impression_change = (April avg impressions/day - March avg impressions/day) / March avg impressions/day`

**Ranking** combines predicted decline risk and predicted decline severity to prioritize a fixed top-50 review queue. The tuned scoring form is:

`p_decline^gamma × (1 + lambda × normalized_predicted_decline_severity)`

with `gamma = 2` and `lambda = 1`.

The target is constructed only after the March feature set has been fixed.

### Five-feature contract

The final model uses exactly five features, all knowable by the decision cutoff:

1. **`aggregate_ctr`** — March click efficiency;
2. **`median_position`** — typical March search position;
3. **`position_slope_per_day`** — March rank movement;
4. **`position_iqr`** — March rank stability/volatility;
5. **`content_age_days`** — lifecycle age measured at 31 March 2026.

Feature selection is intentionally simple and target-blind: candidate variables were grouped into behavioral families, obvious duplicates and unsafe fields were removed using coverage, interpretability and within-family redundancy, and April outcomes were not used to choose the five predictors.

### Frozen baselines

Three baselines were fixed before learned-model comparison:

- **Classification:** training-set decline prior / majority-class prediction;
- **Regression:** training-set mean future change;
- **Ranking:** a transparent low-CTR-for-position rule with a staleness boost.

All learned models are compared with their corresponding baseline on the same evaluation population and metric.

### Validation design

The 2,520-page POC is split by client using `GroupShuffleSplit(test_size=0.25, random_state=42)`:

- development side: **1,800 pages from 15 clients**;
- sealed stress-test side: **720 pages from 6 clients**;
- client overlap: **zero**.

Model and hyperparameter selection occur only on the 15 development clients using **5-fold client-grouped cross-validation**. The six held-out clients are then used as a transfer/generalization stress test and are not used to select the model.

This distinction is central to the paper: grouped development-CV performance and six-client stress-test performance are reported separately.

### Model families

The selected classifier is a **Random Forest classifier**. The selected regressor is a **Random Forest regressor**. Ranking is a tuned risk-severity blend using their outputs.

The project metric contract is:

- classification: ROC-AUC primary; Precision, Recall and F1 secondary;
- regression: RMSE primary; MAE, Median Absolute Error and R² secondary;
- ranking: Precision@50 primary; Recall@50, Lift@50 and NDCG@50 secondary.

### Leakage controls

Every active feature was audited for availability by 31 March 2026, target derivation, future overlap, decision derivation and identifier leakage. All five passed, with no forbidden-feature overlap.

A deliberate leakage experiment demonstrates why this matters. When an exact copy of the future outcome was added as an illegal feature, grouped-holdout regression R² jumped from **-0.142 to 1.000**. A second target-derived classification leakage test increased grouped-CV ROC-AUC from **0.665 to 1.000**. The illegal fields were then removed.

These experiments are diagnostics, not model results; they show how future information can make a weak analysis look artificially perfect.

### Assumptions and interpretation boundary

This is observational prediction and prioritization, not an intervention study. The analysis can support statements about measured associations, out-of-sample ranking performance under the tested splits, and which pages merit review. It does not establish that refreshing a page, changing a title or improving CTR will cause a performance recovery.

In [3]:
with open(repo_path("work/outputs/assignment6_model_benchmark.json"), "r", encoding="utf-8") as fh:
    model_benchmark = json.load(fh)

active_features = leakage_audit["active_features"]
expected_features = [
    "aggregate_ctr",
    "median_position",
    "position_slope_per_day",
    "position_iqr",
    "content_age_days",
]

method_checks = {
    "five_features_exact": active_features == expected_features,
    "no_forbidden_overlap": leakage_audit["forbidden_feature_overlap"] == [],
    "zero_client_overlap": split_manifest["client_overlap"] == [],
    "development_clients": split_manifest["train_clients"] and len(split_manifest["train_clients"]) == 15,
    "stress_test_clients": split_manifest["test_clients"] and len(split_manifest["test_clients"]) == 6,
    "ranking_k_50": split_manifest["ranking_k"] == 50,
    "ranking_gamma_2": model_benchmark["ranking"]["model"]["gamma"] == 2,
    "ranking_lambda_1": model_benchmark["ranking"]["model"]["lambda"] == 1,
}

assert all(method_checks.values()), method_checks
assert all(row["verdict"] == "PASS" for row in leakage_audit["feature_audit"])
assert leakage_audit["deliberate_leakage_test"]["illegal_target_derived_feature_grouped_cv_roc_auc"] == 1

method_summary = pd.DataFrame([
    {"component": "Classification", "selected_method": model_benchmark["classification"]["model"]["name"], "primary_metric": "ROC-AUC"},
    {"component": "Regression", "selected_method": model_benchmark["regression"]["model"]["name"], "primary_metric": "RMSE"},
    {"component": "Ranking", "selected_method": model_benchmark["ranking"]["model"]["name"], "primary_metric": "Precision@50"},
])

display(method_summary)
print("Active features:", active_features)
print("Methodology checks:", method_checks)
print(
    "Leakage demonstration ROC-AUC:",
    round(leakage_audit["deliberate_leakage_test"]["legal_grouped_cv_roc_auc"], 3),
    "->",
    round(leakage_audit["deliberate_leakage_test"]["illegal_target_derived_feature_grouped_cv_roc_auc"], 3),
)

,component,selected_method,primary_metric
0,Classification,RandomForestClassifier,ROC-AUC
1,Regression,RandomForestRegressor,RMSE
2,Ranking,grouped_cv_tuned_risk_severity_blend,Precision@50


Active features: ['aggregate_ctr', 'median_position', 'position_slope_per_day', 'position_iqr', 'content_age_days']
Methodology checks: {'five_features_exact': True, 'no_forbidden_overlap': True, 'zero_client_overlap': True, 'development_clients': True, 'stress_test_clients': True, 'ranking_k_50': True, 'ranking_gamma_2': True, 'ranking_lambda_1': True}
Leakage demonstration ROC-AUC: 0.665 -> 1.0


## 4. Results — model vs baseline

The results are reported in two layers because they answer different questions.

### 4.1 Grouped development validation

Model selection and tuning used five-fold client-grouped cross-validation on the **15 development clients only**. Under that design, the learned components improved on their frozen baselines for all three benchmarked tasks:

| Task | Metric | CV baseline | Learned CV | Reading |
|---|---:|---:|---:|---|
| Classification | ROC-AUC | 0.500 | **0.665** | better discrimination during grouped development CV |
| Regression | RMSE ↓ | 0.852 | **0.806** | lower error during grouped development CV |
| Ranking | Precision@50 | 0.824 | **0.872** | higher top-50 precision during grouped development CV |

These values are **development evidence**, not final unseen-client performance.

### 4.2 Six-client stress test

The selected models were then evaluated once on the sealed set of **720 pages from 6 clients that did not appear in model selection**.

| Task | Metric | Frozen baseline | Learned model | Reading |
|---|---:|---:|---:|---|
| Classification | ROC-AUC | **0.500** | 0.494 | did not beat the prior-probability baseline |
| Regression | RMSE ↓ | 1.431 | **1.379** | slightly lower error than the mean baseline |
| Ranking | Precision@50 | **0.480** | 0.360 | underperformed the transparent ranking rule |

The stress test therefore changes the interpretation materially. The system showed promising grouped-development performance, but that improvement did **not** transfer consistently to unseen clients.

The regression component retained a modest RMSE improvement. The classifier was essentially at chance-level discrimination on the six-client stress test, and the learned top-50 ranking contained **18 true future declines and 32 false picks**, compared with a frozen rule baseline Precision@50 of 0.48.

### 4.3 Validation-design audit

Assignment 7 compared a naive random-page CV with client-grouped CV on the development population.

- random-page classification ROC-AUC: **0.725**
- grouped-client classification ROC-AUC: **0.665**
- random-page ranking Precision@50: **0.948**
- grouped-client ranking Precision@50: **0.872**

Random-page validation allowed the same 15 clients to appear on both sides of every fold; grouped validation reduced that overlap to zero. The more optimistic random-page classification/ranking results therefore illustrate why client separation matters.

### Result interpretation

The headline finding is not that the model universally outperformed the rule baseline. It is that **performance was validation-sensitive**. The learned system looked useful during client-grouped development validation, but the six-client stress test exposed a transfer problem—especially for the primary product metric, Precision@50.

For this reason the downstream action playbook is framed as a **guarded human-review POC**, and fresh validation is required before operational use on materially new client populations.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

with open(repo_path("work/outputs/assignment6_model_benchmark.json"), "r", encoding="utf-8") as fh:
    benchmark = json.load(fh)

with open(repo_path("work/outputs/assignment7_split_audit.json"), "r", encoding="utf-8") as fh:
    split_audit = json.load(fh)

with open(repo_path("work/outputs/assignment6_error_audit.json"), "r", encoding="utf-8") as fh:
    error_audit = json.load(fh)

# Exact paper table: development CV and held-out six-client stress test.
results_table = pd.DataFrame([
    {
        "validation_layer": "Grouped development CV",
        "task": "Classification",
        "metric": "ROC-AUC",
        "baseline": benchmark["cv_baselines"]["summary"]["classification"]["mean"],
        "learned": benchmark["classification"]["model"]["grouped_cv_roc_auc"],
        "better_direction": "higher",
    },
    {
        "validation_layer": "Grouped development CV",
        "task": "Regression",
        "metric": "RMSE",
        "baseline": benchmark["cv_baselines"]["summary"]["regression"]["mean"],
        "learned": benchmark["regression"]["model"]["grouped_cv_rmse"],
        "better_direction": "lower",
    },
    {
        "validation_layer": "Grouped development CV",
        "task": "Ranking",
        "metric": "Precision@50",
        "baseline": benchmark["cv_baselines"]["summary"]["ranking"]["mean"],
        "learned": benchmark["ranking"]["model"]["grouped_cv_precision_at_50"],
        "better_direction": "higher",
    },
    {
        "validation_layer": "Six-client stress test",
        "task": "Classification",
        "metric": "ROC-AUC",
        "baseline": benchmark["classification"]["baseline"]["roc_auc"],
        "learned": benchmark["classification"]["model"]["roc_auc"],
        "better_direction": "higher",
    },
    {
        "validation_layer": "Six-client stress test",
        "task": "Regression",
        "metric": "RMSE",
        "baseline": benchmark["regression"]["baseline"]["rmse"],
        "learned": benchmark["regression"]["model"]["rmse"],
        "better_direction": "lower",
    },
    {
        "validation_layer": "Six-client stress test",
        "task": "Ranking",
        "metric": "Precision@50",
        "baseline": benchmark["ranking"]["baseline"]["precision_at_50"],
        "learned": benchmark["ranking"]["model"]["precision_at_50"],
        "better_direction": "higher",
    },
])

def beats_baseline(row):
    if row["better_direction"] == "higher":
        return row["learned"] > row["baseline"]
    return row["learned"] < row["baseline"]

results_table["learned_beats_baseline"] = results_table.apply(beats_baseline, axis=1)

display(
    results_table[
        ["validation_layer", "task", "metric", "baseline", "learned", "learned_beats_baseline"]
    ].round(4)
)

# Validation audit summary: random-page CV vs honest client-grouped CV.
split_summary = pd.DataFrame(split_audit["summary"]).set_index("split")
validation_audit_table = pd.DataFrame([
    {
        "metric": "Classification ROC-AUC",
        "random_page_cv": split_summary.loc["random_page_cv", "classification_roc_auc"],
        "client_grouped_cv": split_summary.loc["grouped_client_cv", "classification_roc_auc"],
    },
    {
        "metric": "Regression RMSE",
        "random_page_cv": split_summary.loc["random_page_cv", "regression_rmse"],
        "client_grouped_cv": split_summary.loc["grouped_client_cv", "regression_rmse"],
    },
    {
        "metric": "Ranking Precision@50",
        "random_page_cv": split_summary.loc["random_page_cv", "ranking_precision_at_50"],
        "client_grouped_cv": split_summary.loc["grouped_client_cv", "ranking_precision_at_50"],
    },
])

print("\nValidation-design audit:")
display(validation_audit_table.round(4))

# Core assertions: paper must preserve the unfavorable stress-test findings.
stress = results_table[results_table["validation_layer"] == "Six-client stress test"]
assert bool(
    stress.loc[stress["task"] == "Classification", "learned_beats_baseline"].iloc[0]
) is False
assert bool(
    stress.loc[stress["task"] == "Regression", "learned_beats_baseline"].iloc[0]
) is True
assert bool(
    stress.loc[stress["task"] == "Ranking", "learned_beats_baseline"].iloc[0]
) is False

assert error_audit["ranking"]["top50_true_declines"] == 18
assert error_audit["ranking"]["top50_false_picks"] == 32
assert split_summary.loc["random_page_cv", "mean_client_overlap"] == 15
assert split_summary.loc["grouped_client_cv", "mean_client_overlap"] == 0

# Paper-ready comparison figure.
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.5))

tasks = [
    ("Classification", "ROC-AUC", "higher is better"),
    ("Regression", "RMSE", "lower is better"),
    ("Ranking", "Precision@50", "higher is better"),
]

for ax, (task, metric, direction_note) in zip(axes, tasks):
    task_rows = results_table[results_table["task"] == task]
    dev = task_rows[task_rows["validation_layer"] == "Grouped development CV"].iloc[0]
    stress_row = task_rows[task_rows["validation_layer"] == "Six-client stress test"].iloc[0]

    x = np.array([0, 1])
    width = 0.34
    baseline_vals = [dev["baseline"], stress_row["baseline"]]
    learned_vals = [dev["learned"], stress_row["learned"]]

    b1 = ax.bar(x - width/2, baseline_vals, width, label="Baseline")
    b2 = ax.bar(x + width/2, learned_vals, width, label="Learned")

    ax.set_xticks(x)
    ax.set_xticklabels(["Grouped\nCV", "6-client\nstress"])
    ax.set_title(f"{task}\n{metric} ({direction_note})")
    ax.grid(axis="y", alpha=0.2)

    for bars in [b1, b2]:
        ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)

axes[0].set_ylabel("Metric value")
axes[2].legend(loc="best")
fig.suptitle("Model vs frozen baseline: development validation and held-out client stress test", y=1.02)
fig.tight_layout()

figures_dir = repo_path("work/figures")
figures_dir.mkdir(parents=True, exist_ok=True)
figure_path = figures_dir / "capstone_model_vs_baseline.png"
fig.savefig(figure_path, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

print(f"\nSaved paper figure: {figure_path}")
print("Stress-test truth preserved: classification=False, regression=True, ranking=False")

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
